# Day 2 — Baseline + Foundation Model Benchmarks

Evaluates all Day 1–2 models on the Replogle K562 Essential dataset:
- **MeanPredictor** — per-perturbation mean expression
- **LinearAdditive+DEGweight** — control + DEG-frequency-weighted delta
- **KNN** — k-nearest-neighbours in PCA-reduced pseudo-bulk space
- **GEARS** — GO-graph GNN perturbation model
- **scGPT (frozen+δhead)** — scGPT gene embeddings + trainable delta head

All 7 Cell-Eval metrics computed; ranked by mean rank (Arc Generalist Prize logic).

In [ ]:
import sys
sys.path.insert(0, "..")

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

print("PyTorch:", torch.__version__)
print("MPS available:", torch.backends.mps.is_available())

## 1. Load Data

In [ ]:
from data.download import download_replogle_k562
from data.preprocess import preprocess, make_pseudobulk
from data.splits import build_dataset

adata_raw = download_replogle_k562()
print(f"Raw: {adata_raw.n_obs} cells × {adata_raw.n_vars} genes")
print("obs columns:", adata_raw.obs.columns.tolist())

In [ ]:
adata = preprocess(adata_raw, n_hvg=2000)
pseudobulk, pert_col, control_key = make_pseudobulk(adata)

print(f"Pseudo-bulk shape: {pseudobulk.shape}")
print(f"Perturbation column: {pert_col!r}, Control key: {control_key!r}")
print(f"Sample perturbations: {list(pseudobulk.index[:5])}")

In [ ]:
dataset = build_dataset(pseudobulk, control_key=control_key, seed=42)
print(f"Train: {len(dataset.train_perts)} | Val: {len(dataset.val_perts)} | Test: {len(dataset.test_perts)}")

## 2. QC Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# UMI counts per cell
import scanpy as sc
sc.pp.calculate_qc_metrics(adata_raw, inplace=True)
axes[0].hist(adata_raw.obs["total_counts"], bins=50, color="steelblue", alpha=0.8)
axes[0].set_xlabel("Total UMI counts"); axes[0].set_title("UMI distribution")

# Genes per cell
axes[1].hist(adata_raw.obs["n_genes_by_counts"], bins=50, color="salmon", alpha=0.8)
axes[1].set_xlabel("Genes detected"); axes[1].set_title("Gene detection per cell")

# Cells per perturbation
cells_per_pert = adata_raw.obs[pert_col].value_counts()
axes[2].hist(cells_per_pert.values, bins=40, color="mediumseagreen", alpha=0.8)
axes[2].set_xlabel("Cells"); axes[2].set_title("Cells per perturbation")

plt.tight_layout()
plt.savefig("../results/figures/qc_plots.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Median cells per perturbation: {cells_per_pert.median():.0f}")

## 3. Fit Baseline Models

In [ ]:
from models.baselines.mean_predictor import MeanPredictor
from models.baselines.linear_additive import LinearAdditivePredictor
from models.baselines.knn_predictor import KNNPredictor

mean_pred = MeanPredictor()
mean_pred.fit(dataset)

lin_deg = LinearAdditivePredictor(use_deg_weights=True)
lin_deg.fit(dataset)

lin_plain = LinearAdditivePredictor(use_deg_weights=False)
lin_plain.fit(dataset)

knn = KNNPredictor(k=5, n_pca=50)
knn.fit(dataset)

print("Baseline models fitted.")

## 4. Fit GEARS

GEARS trains a GO-graph GNN on the raw single-cell data. Training ~1–2h on M3 CPU.
Skip this cell and use `include_gears=False` if you want faster results.

In [ ]:
include_gears = True  # Set False to skip GEARS training

if include_gears:
    from models.foundation.gears_wrapper import GEARSWrapper

    # Detect perturbation column in raw adata
    from data.preprocess import _find_pert_col, _find_control_key
    raw_pert_col = _find_pert_col(adata_raw)
    raw_ctrl_key = _find_control_key(adata_raw.obs[raw_pert_col])

    gears_model = GEARSWrapper(
        adata_raw=adata,          # preprocessed (log1p HVG), still cell-level
        pert_col=pert_col,
        control_key=control_key,
        epochs=20,
        device="cpu",             # GEARS more stable on CPU; MPS may have issues
    )
    gears_model.fit(dataset)
    print("GEARS fitted.")
else:
    gears_model = None
    print("Skipping GEARS.")

## 5. Fit scGPT Delta Head

Requires `data/checkpoints/scGPT_human/` — run `scripts/download_checkpoints.sh` first.
If checkpoint missing, wrapper falls back to MeanPredictor automatically.

In [ ]:
from models.foundation.scgpt_wrapper import scGPTWrapper

device = "mps" if torch.backends.mps.is_available() else "cpu"
scgpt_model = scGPTWrapper(
    n_epochs=10,
    lr=1e-3,
    batch_size=64,
    device=device,
)
scgpt_model.fit(dataset)
print(f"scGPT wrapper fitted (device={device}, fitted={scgpt_model._fitted}).")

## 6. Evaluate All Models — Full 7-Metric Scorecard

In [ ]:
from eval.harness import ModelEvaluator

models = {
    "MeanPredictor": mean_pred,
    "LinearAdditive+DEGweight": lin_deg,
    "LinearAdditive": lin_plain,
    "KNN(k=5)": knn,
    "scGPT(frozen+δhead)": scgpt_model,
}
if gears_model is not None:
    models["GEARS"] = gears_model

evaluator = ModelEvaluator(dataset, split="test")
leaderboard = evaluator.run_leaderboard(models)
evaluator.save_leaderboard(leaderboard)

print("\n=== Leaderboard ===")
leaderboard.round(4)

## 7. Visualisations

In [ ]:
from eval.leaderboard import generate_leaderboard
generate_leaderboard(save_plots=True)

In [ ]:
# Metric heatmap — which model wins on which metric?
import seaborn as sns

metric_cols = [c for c in leaderboard.columns if c in ["PDS","DES","MAE","PDC","SLC","AUP","SES"]]
heat_data = leaderboard[metric_cols].copy()

# Normalise each metric to [0,1] (higher = better, flip MAE)
normed = heat_data.copy()
for col in metric_cols:
    mn, mx = heat_data[col].min(), heat_data[col].max()
    rng = mx - mn
    if rng < 1e-10:
        normed[col] = 0.5
    elif col == "MAE":
        normed[col] = 1 - (heat_data[col] - mn) / rng  # flip: lower MAE = better
    else:
        normed[col] = (heat_data[col] - mn) / rng

fig, ax = plt.subplots(figsize=(10, max(3, len(normed) * 0.7)))
sns.heatmap(
    normed, annot=heat_data.round(3), fmt=".3f",
    cmap="RdYlGn", vmin=0, vmax=1,
    linewidths=0.5, ax=ax, cbar_kws={"label": "normalised score (higher=better)"}
)
ax.set_title("Cell-Eval 7-Metric Comparison (normalised)", fontsize=13)
plt.tight_layout()
plt.savefig("../results/figures/metric_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Hypothesis H1 Check

> *Zero-shot foundation models will not consistently outperform mean-expression baseline on PDS and DES.*

In [ ]:
if "PDS" in leaderboard.columns and "DES" in leaderboard.columns:
    baseline_pds = leaderboard.loc["MeanPredictor", "PDS"]
    baseline_des = leaderboard.loc["MeanPredictor", "DES"]
    print(f"MeanPredictor baseline — PDS: {baseline_pds:.4f}, DES: {baseline_des:.4f}")
    print()
    for model in leaderboard.index:
        if model == "MeanPredictor":
            continue
        pds = leaderboard.loc[model, "PDS"]
        des = leaderboard.loc[model, "DES"]
        pds_beats = "✓" if pds > baseline_pds else "✗"
        des_beats = "✓" if des > baseline_des else "✗"
        print(f"  {model:35s}  PDS {pds_beats} {pds:.4f}  DES {des_beats} {des:.4f}")
    print()
    print("H1: models marked ✗ on PDS or DES fail to beat the mean predictor — supporting H1.")